# EDA with SQL — Falcon 9 Launch Data

This notebook demonstrates the SQL analyses in the IBM capstone using an in-memory SQLite copy of the frozen 90-record course modelling dataset. The SQL statements use standard SQL syntax. This is a reproducible local companion to the IBM Db2 lab, not a claim that these queries were run on the learner's Db2 service.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display

DATA=Path('data/dataset_part_2.csv')
if not DATA.exists(): raise FileNotFoundError(DATA)
df=pd.read_csv(DATA)
df.columns=[str(c).strip() for c in df.columns]
required={'LaunchSite','PayloadMass','Orbit','Class','Date','BoosterVersion','Outcome'}
missing=required-set(df.columns)
if missing: raise ValueError(f'Missing SQL columns: {sorted(missing)}')
df['PayloadMass']=pd.to_numeric(df['PayloadMass'],errors='coerce')
df['Class']=pd.to_numeric(df['Class'],errors='coerce')
df['Date']=pd.to_datetime(df['Date'],errors='coerce').dt.strftime('%Y-%m-%d')
con=sqlite3.connect(':memory:')
df.to_sql('SPACEXTBL',con,index=False,if_exists='replace')
print('Loaded rows into SPACEXTBL:',len(df))

def query(sql):
    result=pd.read_sql_query(sql,con)
    display(result)
    return result

## 1. Identify unique launch sites

In [ ]:
q1=query('SELECT DISTINCT LaunchSite FROM SPACEXTBL ORDER BY LaunchSite;')

## 2. Count launches by site and calculate observed landing success rate

In [ ]:
q2=query('''SELECT LaunchSite, COUNT(*) AS Launches,
       ROUND(100.0 * AVG(Class), 1) AS SuccessRatePct
FROM SPACEXTBL GROUP BY LaunchSite ORDER BY Launches DESC;''')

## 3. Calculate total and average payload by orbit

In [ ]:
q3=query('''SELECT Orbit, COUNT(PayloadMass) AS PayloadRecords,
       ROUND(SUM(PayloadMass), 1) AS TotalPayloadKg,
       ROUND(AVG(PayloadMass), 1) AS AveragePayloadKg
FROM SPACEXTBL GROUP BY Orbit ORDER BY TotalPayloadKg DESC;''')

## 4. Calculate average payload by booster version

In [ ]:
q4=query('''SELECT BoosterVersion, COUNT(*) AS Launches,
       ROUND(AVG(PayloadMass),1) AS AveragePayloadKg
FROM SPACEXTBL GROUP BY BoosterVersion ORDER BY AveragePayloadKg DESC;''')

## 5. Identify the earliest successful landing record

This query uses the binary Class field and date. It is distinct from the overall mission-success measure.

In [ ]:
q5=query('''SELECT MIN(Date) AS EarliestSuccessfulLandingDate
FROM SPACEXTBL WHERE Class=1;''')

## 6. Count landing outcomes by outcome description

In [ ]:
q6=query('''SELECT Outcome, COUNT(*) AS Records
FROM SPACEXTBL GROUP BY Outcome ORDER BY Records DESC;''')

## 7. Find booster versions associated with the maximum payload mass

In [ ]:
q7=query('''SELECT DISTINCT BoosterVersion, PayloadMass
FROM SPACEXTBL
WHERE PayloadMass=(SELECT MAX(PayloadMass) FROM SPACEXTBL);''')

## 8. Find unsuccessful landing attempts in 2015 by outcome category

In [ ]:
q8=query('''SELECT BoosterVersion, LaunchSite, Date, Outcome
FROM SPACEXTBL
WHERE Class=0 AND Date >= '2015-01-01' AND Date < '2016-01-01'
ORDER BY Date;''')

## 9. Rank sites by successful landing count

In [ ]:
q9=query('''SELECT LaunchSite, SUM(CASE WHEN Class=1 THEN 1 ELSE 0 END) AS SuccessfulLandings
FROM SPACEXTBL GROUP BY LaunchSite ORDER BY SuccessfulLandings DESC;''')

## 10. Analyze yearly launch volume and landing success

In [ ]:
q10=query('''SELECT SUBSTR(Date,1,4) AS LaunchYear,
       COUNT(*) AS Launches,
       ROUND(100.0*AVG(Class),1) AS SuccessRatePct
FROM SPACEXTBL WHERE Date IS NOT NULL
GROUP BY SUBSTR(Date,1,4) ORDER BY LaunchYear;''')

## Interpretation and scope

Use the returned query tables in the presentation and explain each result in the context of the 90-row modelling dataset. Do not substitute numbers from the separate 101-row Db2 lab dataset; their populations and filters differ.